# Day 9 Tutorial：K 折交叉验证

## Goal

在确定性人工回归数据上实现 5 折轮换，保留逐折 RMSE、均值和样本标准差，并用断言证明每个样本恰好验证一次。这里不使用外部测试集；输出不是 ESOL 或粘合剂性能。


## Setup

只依赖 NumPy、pandas、matplotlib 和 scikit-learn。固定数据、模型和折随机种子，使指标可复现。


In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor

RANDOM_SEED = 42
N_SPLITS = 5

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
})


## Steps

### 1. 创建小数据并冻结折索引

每一行是一个人工样本。目标包含二次项和正弦项，便于浅树产生非零误差。


In [ ]:
x_values = np.linspace(-3.0, 3.0, 30)
X = np.column_stack([x_values, x_values ** 2])
y = 0.4 * x_values ** 2 + np.sin(1.5 * x_values)

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
cv_splits = list(cv.split(X))

validation_fold = np.zeros(len(X), dtype=int)
for fold_number, (_, valid_ids) in enumerate(cv_splits, start=1):
    validation_fold[valid_ids] = fold_number

assignment = pd.DataFrame({
    "sample_id": np.arange(len(X)),
    "x": x_values,
    "validation_fold": validation_fold,
})
display(assignment.head(10))


### 2. 每折创建一个新模型

训练与验证索引只在当前折内使用；循环外没有可被复用的已拟合模型。


In [ ]:
fold_rows = []
validation_counts = np.zeros(len(X), dtype=int)

for fold_number, (fit_ids, valid_ids) in enumerate(cv_splits, start=1):
    model = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_SEED)
    model.fit(X[fit_ids], y[fit_ids])
    valid_prediction = model.predict(X[valid_ids])
    validation_counts[valid_ids] += 1

    fold_rows.append({
        "fold": fold_number,
        "fit_rows": len(fit_ids),
        "valid_rows": len(valid_ids),
        "valid_y_mean": float(y[valid_ids].mean()),
        "valid_rmse": float(root_mean_squared_error(y[valid_ids], valid_prediction)),
    })

fold_metrics = pd.DataFrame(fold_rows)
display(fold_metrics)


### 3. 汇总但不隐藏逐折结果

`std` 使用 pandas 默认的样本标准差（`ddof=1`）。


In [ ]:
cv_summary = pd.DataFrame([{
    "n_folds": len(fold_metrics),
    "rmse_mean": fold_metrics["valid_rmse"].mean(),
    "rmse_sample_std": fold_metrics["valid_rmse"].std(ddof=1),
    "rmse_min": fold_metrics["valid_rmse"].min(),
    "rmse_max": fold_metrics["valid_rmse"].max(),
}])
display(cv_summary.round(4))

ax = fold_metrics.plot.bar(x="fold", y="valid_rmse", legend=False, color="#4C78A8")
ax.axhline(cv_summary.loc[0, "rmse_mean"], color="#E45756", linestyle="--", label="mean")
ax.set(title="Day 9 synthetic tutorial: RMSE by fold", ylabel="validation RMSE")
ax.legend()
plt.tight_layout()
plt.show()


## Checks

这些检查验证索引覆盖、折内隔离、表结构和数值有效性；它们不证明随机 KFold 等于 scaffold split。


In [ ]:
assert len(cv_splits) == N_SPLITS
assert np.all(validation_counts == 1), validation_counts
assert set(validation_fold) == set(range(1, N_SPLITS + 1))
assert len(fold_metrics) == N_SPLITS
assert np.isfinite(fold_metrics["valid_rmse"]).all()

for fit_ids, valid_ids in cv_splits:
    assert set(fit_ids).isdisjoint(set(valid_ids))
    assert len(fit_ids) + len(valid_ids) == len(X)

print("Checks passed: each sample was validated exactly once; no fold overlap.")


## Next Steps

完成 `03_exercises.md` 后，把同一程序迁移到本人实验时再创建 `experiments/day09_cross_validation/`。对分子或配方数据，应先判断是否需要 scaffold 或 `GroupKFold`；不要把本教程人工数据分数复制为研究结果。
